In [ ]:
# pip install ultralytics opencv-python

  Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 25.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 29.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 31.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 32.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.5 MB/s  0:00:00
Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl (80.6 MB)
  Attempting uninstall: torchm╺━━━━━━━━━━━━━━━━━━━━━  6/13 [contourpy]hon]
    Found existing installation: torch 2.10.0━━━━━━━━━━━━━━━━━  6/13 [contourpy]
    Uninstalling torch-2.10.0:╸━━━━━━━━━━━━━━━━━━  7/13 [t

## step 1 - trying tracking

In [2]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")  # start small

cap = cv2.VideoCapture("matches_for_labeling/TEST CLIP 1.mp4")

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        for box, cls in zip(boxes, classes):
            x1, y1, x2, y2 = box
            print(frame_idx, cls, x1, y1, x2, y2)

    frame_idx += 1

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/finneganlaister-smith/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


OpenCV: Couldn't read video stream from file "matches_for_labeling/TEST CLIP 1.mp4"


In [ ]:
#out of the box YOLO 
from ultralytics import YOLO
import cv2
import os

model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture("matches for labeling/TEST CLIP 1.mp4")

if not cap.isOpened():
    print("Error: Could not open video")
    exit()

frame_idx = 0
max_frames = 200

os.makedirs("good_frames", exist_ok=True)

while True:
    ret, frame = cap.read()
    if not ret or frame_idx > max_frames:
        break

    results = model.track(
        frame,
        conf=0.1,
        imgsz=1280,
        classes=[0, 32],
        persist=True
    )

    boxes = results[0].boxes
    num_persons = 0

    if boxes is not None and boxes.cls is not None:
        classes = boxes.cls.cpu().numpy()
        num_persons = (classes == 0).sum()

    if num_persons < 8:
        frame_idx += 1
        continue

    annotated = results[0].plot()

    if frame_idx % 10 == 0:
        cv2.imwrite(f"good_frames/frame_{frame_idx}.jpg", frame)
        cv2.imwrite(f"good_frames/annotated_{frame_idx}.jpg", annotated)

    cv2.imshow("detections", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    print(f"Frame {frame_idx}: {num_persons} persons")
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()


0: 736x1280 1 sports ball, 299.9ms
Speed: 5.5ms preprocess, 299.9ms inference, 1.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 sports ball, 262.1ms
Speed: 3.7ms preprocess, 262.1ms inference, 1.2ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 sports ball, 304.3ms
Speed: 4.0ms preprocess, 304.3ms inference, 1.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 sports ball, 258.4ms
Speed: 3.6ms preprocess, 258.4ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 sports ball, 275.5ms
Speed: 3.8ms preprocess, 275.5ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 sports ball, 260.2ms
Speed: 3.7ms preprocess, 260.2ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 sports ball, 257.8ms
Speed: 3.6ms preprocess, 257.8ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 

In [ ]:
#ByteTrack
from ultralytics import YOLO
import cv2
import os

model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture("matches for labeling/TEST CLIP 2.mp4")

if not cap.isOpened():
    print("Error: Could not open video")
    exit()

frame_idx = 0
max_frames = 200

os.makedirs("good_frames", exist_ok=True)

while True:
    ret, frame = cap.read()
    if not ret or frame_idx > max_frames:
        break

    results = model.track(
        frame,
        conf=0.15,
        imgsz=1280,
        classes=[0],          # just people for now
        persist=True,
        tracker="bytetrack.yaml"
    )

    boxes = results[0].boxes
    num_persons = 0

    # --- Added logic for tracking and printing IDs and positions ---
    if (
        boxes is not None
        and boxes.id is not None
        and boxes.cls is not None
        and boxes.xyxy is not None
    ):
        track_ids = boxes.id.int().cpu().tolist()
        classes = boxes.cls.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        for box, cls, track_id in zip(xyxy, classes, track_ids):
            if int(cls) == 0:
                x1, y1, x2, y2 = box
                print(f"Frame {frame_idx}: player ID {track_id} at [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
        num_persons = int((classes == 0).sum())
    # -------------------------------------------------------------

    elif boxes is not None and boxes.cls is not None:
        classes = boxes.cls.cpu().numpy()
        num_persons = int((classes == 0).sum())

    if num_persons < 8:
        frame_idx += 1
        continue

    annotated = results[0].plot()

    if frame_idx % 10 == 0:
        cv2.imwrite(f"good_frames/frame_{frame_idx}.jpg", frame)
        cv2.imwrite(f"good_frames/annotated_{frame_idx}.jpg", annotated)

    cv2.imshow("detections", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    print(f"Frame {frame_idx}: {num_persons} persons")
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()


0: 736x1280 20 persons, 303.3ms
Speed: 8.0ms preprocess, 303.3ms inference, 7.7ms postprocess per image at shape (1, 3, 736, 1280)
Frame 0: player ID 1 at [554.2, 330.9, 569.7, 372.6]
Frame 0: player ID 2 at [583.8, 309.0, 608.4, 346.0]
Frame 0: player ID 3 at [349.1, 269.8, 364.5, 308.1]
Frame 0: player ID 4 at [221.2, 332.6, 236.7, 376.3]
Frame 0: player ID 5 at [102.2, 267.6, 114.1, 307.4]
Frame 0: player ID 6 at [649.7, 233.2, 663.7, 266.8]
Frame 0: player ID 7 at [737.9, 257.7, 757.6, 289.9]
Frame 0: player ID 8 at [886.5, 356.3, 903.4, 397.8]
Frame 0: player ID 9 at [706.6, 267.7, 720.8, 300.6]
Frame 0: player ID 10 at [676.5, 227.2, 688.2, 257.7]
Frame 0: player ID 11 at [409.8, 213.0, 423.8, 241.5]
Frame 0: player ID 12 at [472.9, 267.0, 485.8, 299.0]
Frame 0: player ID 13 at [618.8, 233.1, 628.5, 263.8]
Frame 0: player ID 14 at [348.3, 210.9, 368.5, 243.3]
Frame 0: player ID 15 at [374.4, 232.5, 390.0, 266.3]
Frame 0: player ID 16 at [608.5, 239.3, 620.5, 271.1]
Frame 0: play

: 

## step 2 - combining  ByteTrack (better than otb) with re-id

In [ ]:
# In a Jupyter notebook, you should install packages like this:
# !pip install tensorboard
#gdown
#torchreid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [tensorboard] [tensorboard]


In [ ]:
#pt 0 - imports and model loading
import torch
import torchreid

reid_model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=1000,
    pretrained=True
)

reid_model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
reid_model.to(device)

Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /Users/finneganlaister-smith/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 50.9MB/s]

Successfully loaded imagenet pretrained weights from "/Users/finneganlaister-smith/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"


OSNet(
  (conv1): ConvLayer(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Sequential(
    (0): OSBlock(
      (conv1): Conv1x1(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (conv2a): LightConv3x3(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (conv2b): Sequential(
        (

In [ ]:
#pt 1 - GET EMBEDDING FROM A PLAYER CROP

def get_embedding(crop):
    crop = cv2.resize(crop, (128, 256))
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    crop = crop.astype(np.float32) / 255.0

    crop = np.transpose(crop, (2, 0, 1))
    crop = torch.tensor(crop).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = reid_model(crop)

    return embedding.cpu().numpy().flatten()


track_embeddings = {}   # track_id -> list of embeddings
lost_tracks = {}        # track_id -> last embedding



####



####


current_ids = set(ids)

for tid in list(track_embeddings.keys()):
    if tid not in current_ids:
        # store last embedding
        lost_tracks[tid] = track_embeddings[tid][-1]
        del track_embeddings[tid]


from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))


if track_id not in track_embeddings:
    emb = get_embedding(crop)

    best_match = None
    best_score = 0

    for lost_id, lost_emb in lost_tracks.items():
        score = cosine_similarity(emb, lost_emb)

        if score > best_score:
            best_score = score
            best_match = lost_id

    if best_score > 0.7:  # threshold to tune
        print(f"Reconnecting {track_id} -> {best_match}")

In [ ]:
#pt 2 - ByteTrack
from ultralytics import YOLO
import cv2
import os

model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture("matches for labeling/TEST CLIP 2.mp4")

if not cap.isOpened():
    print("Error: Could not open video")
    exit()

frame_idx = 0
max_frames = 200

os.makedirs("good_frames", exist_ok=True)

while True:
    ret, frame = cap.read()
    if not ret or frame_idx > max_frames:
        break

    results = model.track(
        frame,
        conf=0.15,
        imgsz=1280,
        classes=[0],          # just people for now
        persist=True,
        tracker="bytetrack.yaml"
    )

    boxes = results[0].boxes
    num_persons = 0

    # --- Added logic for tracking and printing IDs and positions ---
    if (
        boxes is not None
        and boxes.id is not None
        and boxes.cls is not None
        and boxes.xyxy is not None
    ):
        track_ids = boxes.id.int().cpu().tolist()
        classes = boxes.cls.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        for box, cls, track_id in zip(xyxy, classes, track_ids):
            if int(cls) == 0:
                x1, y1, x2, y2 = box
                print(f"Frame {frame_idx}: player ID {track_id} at [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
        num_persons = int((classes == 0).sum())
    # -------------------------------------------------------------

    elif boxes is not None and boxes.cls is not None:
        classes = boxes.cls.cpu().numpy()
        num_persons = int((classes == 0).sum())

    if num_persons < 8:
        frame_idx += 1
        continue

    annotated = results[0].plot()

    if frame_idx % 10 == 0:
        cv2.imwrite(f"good_frames/frame_{frame_idx}.jpg", frame)
        cv2.imwrite(f"good_frames/annotated_{frame_idx}.jpg", annotated)

    cv2.imshow("detections", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    print(f"Frame {frame_idx}: {num_persons} persons")
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()

In [ ]:
#YOLO + ByteTrack + re-id script
from ultralytics import YOLO
import cv2
import os
import torch
import torchreid
import numpy as np
from numpy.linalg import norm

# ---------------------------
# LOAD MODELS
# ---------------------------
model = YOLO("yolov8l.pt") #used yolov8m

reid_model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=1000,
    pretrained=True
)
reid_model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
reid_model.to(device)

# ---------------------------
# FUNCTIONS
# ---------------------------
def get_embedding(crop):
    crop = cv2.resize(crop, (128, 256))
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    crop = crop.astype(np.float32) / 255.0
    crop = np.transpose(crop, (2, 0, 1))
    crop = torch.tensor(crop).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = reid_model(crop)

    return embedding.cpu().numpy().flatten()


def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))


def distance(p1, p2):
    return np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

# ---------------------------
# TRACK STORAGE
# ---------------------------
track_embeddings = {}   # active tracks
lost_tracks = {}        # recently lost tracks
lost_tracks_age = {}    # ages for lost tracks
lost_positions = {}     # store lost track positions
track_lifetime = {}     # track id -> number of frames it has existed

last_positions = {}

MAX_AGE = 30           # Memory window (in frames)

# ---------------------------
# VIDEO
# ---------------------------
cap = cv2.VideoCapture("matches for labeling/TEST CLIP 2.mp4")

if not cap.isOpened():
    print("Error: Could not open video")
    exit()

frame_idx = 0
max_frames = 200

os.makedirs("good_frames", exist_ok=True)

# ---------------------------
# MAIN LOOP
# ---------------------------
while True:
    ret, frame = cap.read()
    if not ret or frame_idx > max_frames:
        break

    results = model.track(
        frame,
        conf=0.10, #0.15
        imgsz=1536,  # Increased from 1280 to 1536 per instruction
        classes=[0],
        persist=True,
        tracker="custom_bytetrack.yaml",  # <--------- ADDED THIS LINE
    )

    boxes = results[0].boxes
    current_ids = set()
    num_persons = 0

    # CLEAN UP OLD LOST TRACKS (before matching)
    for lost_id in list(lost_tracks.keys()):
        if frame_idx - lost_tracks_age[lost_id] > MAX_AGE:
            del lost_tracks[lost_id]
            del lost_tracks_age[lost_id]
            if lost_id in lost_positions:
                del lost_positions[lost_id]

    xyxy = None
    track_ids = None

    if (
        boxes is not None
        and boxes.id is not None
        and boxes.cls is not None
        and boxes.xyxy is not None
    ):
        track_ids = boxes.id.int().cpu().tolist()
        classes = boxes.cls.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for box, cls, track_id in zip(xyxy, classes, track_ids):
            if int(cls) != 0:
                continue

            x1, y1, x2, y2 = map(int, box)
            # --- Begin Insert per instruction ---
            x_center = (x1 + x2) / 2
            y_center = (y1 + y2) / 2
            last_positions[track_id] = (x_center, y_center)
            # --- End Insert per instruction ---

            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            current_ids.add(track_id)
            num_persons += 1

            emb = get_embedding(crop)

            # Update lifetime counter for this track
            track_lifetime[track_id] = track_lifetime.get(track_id, 0) + 1

            # EXISTING TRACK
            if track_id in track_embeddings:
                track_embeddings[track_id].append(emb)
                if len(track_embeddings[track_id]) > 10:
                    track_embeddings[track_id].pop(0)

            # NEW TRACK → Try to reconnect, with cooldown
            else:
                # Only attempt reconnection if track has lived for enough frames
                if track_lifetime[track_id] < 3:
                    track_embeddings[track_id] = [emb]
                    continue

                best_match = None
                best_score = 0
                current_pos = ((x1+x2)/2, (y1+y2)/2)

                for lost_id, lost_emb in lost_tracks.items():
                    # Only match if we know lost position for this lost_id
                    if lost_id in lost_positions:
                        dist = distance(current_pos, lost_positions[lost_id])
                    else:
                        dist = float('inf')

                    score = cosine_similarity(emb, lost_emb)

                    if score > best_score and dist < 120:
                        best_score = score
                        best_match = lost_id

                if best_score > 0.8 and best_match is not None:
                    print(f"Reconnecting new ID {track_id} -> old ID {best_match}")

                    track_embeddings[track_id] = [emb]   # keep the NEW ID
                    del lost_tracks[best_match]
                    del lost_tracks_age[best_match]
                    if best_match in lost_positions:
                        del lost_positions[best_match]

                else:
                    track_embeddings[track_id] = [emb]

    # MOVE LOST TRACKS
    # Need to store the last known position (center) when a track is moved to lost
    # For tracks being moved from active to lost, use the last bounding box seen for that track this frame
    # We'll build a tid->bbox map for quickly getting the bbox of lost track
    last_seen_bboxes = {}
    if xyxy is not None and track_ids is not None:
        for box, tid in zip(xyxy, track_ids):
            last_seen_bboxes[tid] = box

    for tid in list(track_embeddings.keys()):
        if tid not in current_ids:
            lost_tracks[tid] = np.mean(track_embeddings[tid], axis=0)
            lost_tracks_age[tid] = frame_idx

            # Store last position if known
            if tid in last_positions:
                lost_positions[tid] = last_positions[tid]
            del track_embeddings[tid]
            # When lost, we can pop the lifetime
            if tid in track_lifetime:
                del track_lifetime[tid]

    # FILTER LOW PLAYER COUNT
    if num_persons < 8:
        frame_idx += 1
        continue

    # BEGIN REPLACEMENT: annotated/visualization code
    annotated = frame.copy()
    if xyxy is not None and track_ids is not None:
        for box, track_id in zip(xyxy, track_ids):
            x1, y1, x2, y2 = map(int, box)
            # --- Begin Insert per instruction ---
            x_center = (x1 + x2) / 2
            y_center = (y1 + y2) / 2
            last_positions[track_id] = (x_center, y_center)
            # --- End Insert per instruction ---
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(
                annotated,
                f"ID {track_id}",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,255,0),
                2
            )
    # END REPLACEMENT

    if frame_idx % 10 == 0:
        cv2.imwrite(f"good_frames/frame_{frame_idx}.jpg", frame)
        cv2.imwrite(f"good_frames/annotated_{frame_idx}.jpg", annotated)

    cv2.imshow("tracking", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    print(f"Frame {frame_idx}: {num_persons} persons")
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()

/Users/finneganlaister-smith/Downloads/DEV_ENVIRONMENT/data-science-jupyter-template-main/Research/dbpy-env/lib/python3.12/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


Successfully loaded imagenet pretrained weights from "/Users/finneganlaister-smith/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"

0: 864x1536 13 persons, 741.9ms
Speed: 9.2ms preprocess, 741.9ms inference, 7.7ms postprocess per image at shape (1, 3, 864, 1536)
Frame 0: 13 persons

0: 864x1536 13 persons, 668.8ms
Speed: 5.1ms preprocess, 668.8ms inference, 1.2ms postprocess per image at shape (1, 3, 864, 1536)
Frame 1: 13 persons

0: 864x1536 14 persons, 603.3ms
Speed: 4.3ms preprocess, 603.3ms inference, 1.1ms postprocess per image at shape (1, 3, 864, 1536)
Frame 2: 14 persons

0: 864x1536 13 persons, 610.2ms
Speed: 4.9ms preprocess, 610.2ms inference, 1.1ms postprocess per image at shape (1, 3, 864, 1536)
Frame 3: 13 persons

0: 864x1536 16 persons, 602.7ms
Speed: 5.2ms preprocess, 602.7ms inference, 1.1ms postprocess per image at shape (1, 3, 864, 1536)
Frame 4: 16 persons

0: 864x1536 15 persons, 599.6ms
Speed: 4.9ms preprocess, 599.6ms inference, 1.2ms postprocess per image at

: 